# 🫁 Respiratory Sound Analysis

> Analysis of respiratory sounds using advanced signal processing techniques

---

## 📦 Installation & Setup

In [ ]:
!pip install -q kaggle numpy scipy matplotlib pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ⚙️ Configure Kaggle & Download Dataset

In [ ]:
import os
import shutil
from google.colab import files

KAGGLE_JSON_DRIVE = '/content/drive/MyDrive/kaggle.json'
KAGGLE_JSON_LOCAL = os.path.expanduser('~/.kaggle/kaggle.json')

if os.path.exists(KAGGLE_JSON_DRIVE):
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy(KAGGLE_JSON_DRIVE, KAGGLE_JSON_LOCAL)
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
else:
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(KAGGLE_JSON_LOCAL, 'wb') as f:
        f.write(list(uploaded.values())[0])
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
    shutil.copy(KAGGLE_JSON_LOCAL, KAGGLE_JSON_DRIVE)

In [ ]:
import os

DATASET_DRIVE = '/content/drive/MyDrive/respiratory_sound_dataset'
DATASET_LOCAL = '/content/respiratory_sound_dataset'

if os.path.exists(DATASET_DRIVE) and os.listdir(DATASET_DRIVE):
    if not os.path.exists(DATASET_LOCAL):
        os.symlink(DATASET_DRIVE, DATASET_LOCAL)
    print(f"✅ Dataset found in Drive: {len(os.listdir(DATASET_DRIVE))} items")
else:
    print("📥 Downloading dataset...")
    !kaggle datasets download -d vbookshelf/respiratory-sound-database
    os.makedirs(DATASET_DRIVE, exist_ok=True)
    !unzip -q respiratory-sound-database.zip -d {DATASET_DRIVE}
    !rm respiratory-sound-database.zip
    if not os.path.exists(DATASET_LOCAL):
        os.symlink(DATASET_DRIVE, DATASET_LOCAL)
    print(f"✅ Dataset saved to Drive")

## 📥 Clone Analysis Repository

In [ ]:
import sys

REPO_PATH = '/content/course_paper'

if os.path.exists(REPO_PATH):
    print(f"✅ Repository already cloned")
else:
    print("📥 Cloning repository...")
    !git clone --depth 1 --filter=blob:none --sparse https://github.com/incRED1bl/course_paper.git {REPO_PATH}
    !git -C {REPO_PATH} sparse-checkout set app
    print("✅ Repository cloned")

sys.path.insert(0, REPO_PATH)

In [ ]:
import os
import numpy as np
from scipy.io import wavfile
from pathlib import Path

def load_respiratory_sounds(dataset_path, max_files=20):
    """Load audio files from the dataset efficiently."""
    dataset_path = Path(dataset_path)
    
    if not dataset_path.exists():
        raise FileNotFoundError(f"Path not found: {dataset_path}")
    
    audio_dir = None
    for pattern in ['**/audio_and_txt_files', '**/Respiratory_Sound_Database/**/audio_and_txt_files']:
        matches = list(dataset_path.glob(pattern))
        if matches:
            audio_dir = matches[0]
            break
    
    if not audio_dir or not audio_dir.exists():
        raise FileNotFoundError(f"audio_and_txt_files folder not found in {dataset_path}")
    
    wav_files = sorted(audio_dir.glob("*.wav"))
    load_count = min(max_files, len(wav_files))
    
    signals = {}
    load_errors = 0
    
    for wav_file in wav_files[:load_count]:
        try:
            sample_rate, signal_data = wavfile.read(str(wav_file))
            if signal_data.dtype != np.float64:
                signal_data = signal_data.astype(np.float64)
            if signal_data.ndim > 1:
                signal_data = signal_data[:, 0]
            
            signals[wav_file.name] = {
                'signal': signal_data,
                'sample_rate': sample_rate
            }
        except Exception as e:
            load_errors += 1
            if load_errors <= 3:
                print(f"⚠️ Failed: {wav_file.name}")
    
    if load_errors > 3:
        print(f"⚠️ ...and {load_errors - 3} more errors")
    
    return signals

## 🔊 Load Audio Files & Extract Features

In [ ]:
signals = load_respiratory_sounds(DATASET_LOCAL, max_files=20)
sample_rate = list(signals.values())[0]['sample_rate']

print(f"✅ Loaded {len(signals)} files @ {sample_rate} Hz")

In [ ]:
import pandas as pd
from pathlib import Path
from app.data_preprocessing import extract_features_batch

diagnosis_path = Path(DATASET_LOCAL)
diagnosis_file = None

for pattern in ['**/patient_diagnosis.csv', '**/Respiratory_Sound_Database/**/patient_diagnosis.csv']:
    matches = list(diagnosis_path.glob(pattern))
    if matches:
        diagnosis_file = matches[0]
        break

if not diagnosis_file:
    raise FileNotFoundError("patient_diagnosis.csv not found")

diagnosis_df = pd.read_csv(diagnosis_file)
patient_col, diagnosis_col = diagnosis_df.columns[0], diagnosis_df.columns[1]
diagnosis_map = dict(zip(diagnosis_df[patient_col], diagnosis_df[diagnosis_col]))

rows = extract_features_batch(signals, embedding_dim=3, time_delay=1)

for row in rows:
    row['diagnosis'] = diagnosis_map.get(row['patient_id'], 'Unknown')

results_df = pd.DataFrame(rows)

print(f"✅ Extracted features: {results_df.shape}")
print(f"\n📋 Diagnoses:\n{results_df['diagnosis'].value_counts()}")
display(results_df.head())

## 📊 Visualizations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from app.features import extract_frequency_features

sample_files = list(signals.keys())[:3]

fig, axes = plt.subplots(len(sample_files), 2, figsize=(14, 4 * len(sample_files)))
if len(sample_files) == 1:
    axes = axes.reshape(1, -1)

for idx, filename in enumerate(sample_files):
    signal_data = signals[filename]['signal']
    sample_rate_file = signals[filename]['sample_rate']
    
    fig.suptitle('Time & Frequency Domain Analysis', fontsize=16, fontweight='bold', y=0.995)
    
    ax1, ax2 = axes[idx]
    
    n_samples = min(1000, len(signal_data))
    ax1.plot(signal_data[:n_samples], linewidth=0.8, color='steelblue')
    ax1.set_title(f'{filename[:30]}... - Time Domain', fontsize=11)
    ax1.set_xlabel('Samples')
    ax1.set_ylabel('Amplitude')
    ax1.grid(True, alpha=0.3)
    
    frequencies, magnitudes, _ = extract_frequency_features(signal_data, sample_rate_file)
    
    max_freq_idx = np.searchsorted(frequencies, 2000)
    ax2.plot(frequencies[:max_freq_idx], magnitudes[:max_freq_idx], 
             linewidth=1.2, color='steelblue')
    ax2.axvspan(400, 1600, alpha=0.2, color='red', label='Wheeze Range')
    ax2.axvspan(100, 400, alpha=0.1, color='green', label='Normal Breath')
    ax2.set_title('Frequency Spectrum', fontsize=11)
    ax2.set_xlabel('Frequency (Hz)')
    ax2.set_ylabel('Magnitude')
    ax2.grid(True, alpha=0.3)
    if idx == 0:
        ax2.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
n_samples = min(5, len(results_df))
sample_data = results_df.head(n_samples)
energy_cols = ['low_freq_energy', 'mid_freq_energy', 'high_freq_energy']

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(n_samples)
width = 0.25
colors = ['#2ecc71', '#3498db', '#e74c3c']

for i, (col, color) in enumerate(zip(energy_cols, colors)):
    energies = sample_data[col].values
    offset = (i - 1) * width
    ax.bar(x + offset, energies, width, 
           label=col.replace('_', ' ').title(), 
           alpha=0.85, color=color, edgecolor='black', linewidth=0.8)

ax.set_title('Energy Distribution Across Frequency Bands', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Audio File', fontsize=12)
ax.set_ylabel('Normalized Energy', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels([f[:20] + '...' if len(f) > 20 else f 
                    for f in sample_data['filename']], 
                   rotation=25, ha='right')
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(sample_data[energy_cols].max().max() * 1.1, 0.1))

plt.tight_layout()
plt.show()

In [ ]:
n_plot = min(20, len(results_df))
sample_data = results_df.head(n_plot)

fig, ax = plt.subplots(figsize=(10, 8))

diagnosis_colors = {
    'COPD': '#e74c3c',
    'Healthy': '#2ecc71', 
    'Asthma': '#3498db',
    'URTI': '#f39c12',
    'Bronchiectasis': '#9b59b6',
    'Pneumonia': '#e67e22',
    'Bronchiolitis': '#1abc9c',
    'LRTI': '#34495e'
}

for diagnosis in sample_data['diagnosis'].unique():
    subset = sample_data[sample_data['diagnosis'] == diagnosis]
    color = diagnosis_colors.get(diagnosis, '#95a5a6')
    ax.scatter(subset['entropy'], subset['complexity'], 
              s=150, alpha=0.7, c=color, label=diagnosis,
              edgecolors='black', linewidths=1.2)

annotation_indices = [0, n_plot//2, min(n_plot-1, len(sample_data)-1)]
for i in annotation_indices:
    if i < len(sample_data):
        row = sample_data.iloc[i]
        ax.annotate(row['filename'][:12] + '...', 
                   (row['entropy'], row['complexity']), 
                   xytext=(8, 8), textcoords='offset points', 
                   fontsize=8, alpha=0.6,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                            alpha=0.7, edgecolor='gray'))

ax.set_title('Entropy-Complexity Plane (H×C)', fontsize=14, fontweight='bold')
ax.set_xlabel('Normalized Entropy (H)', fontsize=12)
ax.set_ylabel('Statistical Complexity (C)', fontsize=12)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='best', fontsize=10, framealpha=0.9)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, max(sample_data['complexity'].max() * 1.1, 0.5))

plt.tight_layout()
plt.show()